Load Packages

In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import Polygon
import math

from compactness_metric_functions import *

import maup

import networkx as nx

from gerrychain import (GeographicPartition, Partition, Graph, MarkovChain,
                        proposals, updaters, constraints, accept, Election)
from gerrychain.proposals import recom, propose_random_flip
from functools import partial
import pandas
from gerrychain.tree import recursive_tree_part

from gerrychain.metrics import efficiency_gap, mean_median, polsby_popper, partisan_bias

import os

from gerrychain.constraints.contiguity import contiguous_components, contiguous

from gerrychain.updaters import cut_edges

from gerrychain.tree import bipartition_tree, find_balanced_edge_cuts_memoization

from gerrychain.updaters import num_spanning_trees

import numpy as np

import pandas as pd

import random

Read Files

In [2]:
CON = gpd.read_file("./NC_Processed/nc_cong_adopted_2025/NCGA_CCM-2 - Shapefile/CCM-2.shp")
Precincts = gpd.read_file("./NC_Processed/output/NC_Processed_Precincts_ALARM.shp")

graph = Graph.from_json("./NC_Processed/output/NC_Processed_Precincts_ALARM.json")


# Compactness
Setup

In [78]:
Precincts.keys()

Index(['GEOID', 'state', 'county', 'muni', 'cnty_mn', 'cd_2010', 'vtd', 'pop',
       'pop_hsp', 'pop_wht', 'pp_blck', 'pop_ain', 'pop_asn', 'pop_nhp',
       'pop_thr', 'pop_two', 'vap', 'vap_hsp', 'vap_wht', 'vp_blck', 'vap_ain',
       'vap_asn', 'vap_nhp', 'vap_thr', 'vap_two', 'pr_16_r_', 'pr_16_d_',
       'uss_16_r_', 'uss_16_d_', 'gv_16_r_', 'gv_16_d_', 'atg_16_r_',
       'atg_16_d_', 'ss_16_r_', 'ss_16_d_', 'pr_20_r_', 'pr_20_d_',
       'uss_20_r_', 'uss_20_d_', 'gv_20_r_', 'gv_20_d_', 'atg_20_r_',
       'atg_20_d_', 'ss_20_r_', 'ss_20_d_', 'arv_16', 'adv_16', 'arv_18',
       'adv_18', 'arv_20', 'adv_20', 'nrv', 'ndv', 'are_lnd', 'are_wtr',
       'cd_2020', 'psd_cnt', 'CON', 'SLDU', 'SLDL', 'C_X', 'C_Y', 'geometry'],
      dtype='object')

In [3]:
def polsby_popper_gdf(gdf):
    return 4*math.pi * gdf.area/(gdf.length**2)

def count_spanning(graph):
    laplacian = nx.laplacian_matrix(graph)
    L = np.delete(np.delete(laplacian.todense(), 0, 0), 1, 1)
    return np.linalg.slogdet(L)[1]

def county_splits(partition, df=Precincts):
    df["current"] = df.index.map(partition.assignment)

    counties = sum(df.groupby("county")['current'].nunique()>1)
    return counties

election_names = [
    "PRE",
    "USS",
    "GOV"
]

num_elections = len(election_names)

election_columns = [
  ['pr_20_r_','pr_20_d_'],
  ['uss_20_r_','uss_20_d_'],
  ['gv_20_r_','gv_20_d_']
]

my_updaters = {
    "population": updaters.Tally("pop", alias="population"),
    "cut_edges": cut_edges,
    "PP":polsby_popper,
    "county_splits": county_splits
}

elections = [
    Election(
        election_names[i],
        {"Democratic": election_columns[i][1], "Republican": election_columns[i][0]},
    )
    for i in range(num_elections)
]

election_updaters = {election.name: election for election in elections}
for node in graph.nodes():
    graph.nodes()[node]["non_pp_blck"] = graph.nodes()[node]["pop"] - graph.nodes()[node]["pp_blck"]

my_updaters.update({"pp_blck": Election("pp_blck", {"pp_blck": "pp_blck", "non_pp_blck": "non_pp_blck"})})

# save percentages

my_updaters.update(election_updaters)


## District-level compactness scores

In [4]:
CON_from_Precincts = Precincts.dissolve(by='CON')
SLDU_from_Precincts = Precincts.dissolve(by='SLDU')
SLDL_from_Precincts = Precincts.dissolve(by='SLDL')

In [5]:
plans = [CON_from_Precincts,SLDU_from_Precincts,SLDL_from_Precincts]

for plan in plans: 
    plan['% Black'] = plan['pp_blck']/plan['pop']
    plan['PP'] = polsby_popper_gdf(plan)
    plan['CH'] = c_hull_ratio(plan)
    plan['R'] = 0
    plan['LW'] = 0
    plan['P'] = plan.length
    for ind, row in plan.iterrows():
        plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
        
        
        outside = list(row['geometry'].convex_hull.envelope.exterior.coords)

        o_len = max([x[0] for x in outside]) - min([x[0] for x in outside])

        o_wid = max([x[1] for x in outside]) - min([x[1] for x in outside])

        lw = min(o_len/o_wid,o_wid/o_len)
        
        plan.loc[ind,'LW'] = lw

C:\Users\angel\AppData\Local\Temp\ipykernel_26804\2582077219.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.49165906290067457' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
C:\Users\angel\AppData\Local\Temp\ipykernel_26804\2582077219.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.7749329453819488' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'LW'] = lw
C:\Users\angel\AppData\Local\Temp\ipykernel_26804\2582077219.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.26253421579831393' has dtype incompatible 

In [6]:
CON_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_CON_ALARM.csv")
SLDU_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_SLDU_ALARM.csv")
SLDL_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./NC_Stats/NC_Compactness_SLDL_ALARM.csv")

## Cut Edges

In [4]:
CONPart = GeographicPartition(graph,"CON",my_updaters)
SLDUPart = GeographicPartition(graph,"SLDU",my_updaters)
SLDLPart = GeographicPartition(graph,"SLDL",my_updaters)

In [5]:
len(SLDUPart.parts.keys())

50

In [15]:
for i in range(120):
    if not nx.is_connected(graph.subgraph(SLDLPart.parts[i])):
        print(i)

In [85]:
"""
#for i in range(50):
#    print(i,nx.is_connected(graph.subgraph(SLDUPart.parts[i])))

# 5, 28

subgraph5 = graph.subgraph(SLDUPart.parts[5])
subgraph28 = graph.subgraph(SLDUPart.parts[28])

S5 = [c for c in nx.connected_components(subgraph5)]
S28 = [c for c in nx.connected_components(subgraph28)]

print([len(x) for x in S5])
print([len(x) for x in S28])

print(S5[0])
print(S28[0])

print(graph.nodes()[2433])


graph.nodes()[284]["SLDU"] = 27
graph.nodes()[2475]["SLDU"] = 5

S5 = [c for c in nx.connected_components(subgraph5)]
S28 = [c for c in nx.connected_components(subgraph28)]

print([len(x) for x in S5])
print([len(x) for x in S28])

#print(graph.nodes()[2433])

for node in graph.nodes():
    if graph.nodes()[node]["GEOID"] == '37183017-02':
        print(node)

#graph.nodes()[1799]
"""

'\n#for i in range(50):\n#    print(i,nx.is_connected(graph.subgraph(SLDUPart.parts[i])))\n\n# 5, 28\n\nsubgraph5 = graph.subgraph(SLDUPart.parts[5])\nsubgraph28 = graph.subgraph(SLDUPart.parts[28])\n\nS5 = [c for c in nx.connected_components(subgraph5)]\nS28 = [c for c in nx.connected_components(subgraph28)]\n\nprint([len(x) for x in S5])\nprint([len(x) for x in S28])\n\nprint(S5[0])\nprint(S28[0])\n\nprint(graph.nodes()[2433])\n\n\ngraph.nodes()[284]["SLDU"] = 27\ngraph.nodes()[2475]["SLDU"] = 5\n\nS5 = [c for c in nx.connected_components(subgraph5)]\nS28 = [c for c in nx.connected_components(subgraph28)]\n\nprint([len(x) for x in S5])\nprint([len(x) for x in S28])\n\n#print(graph.nodes()[2433])\n\nfor node in graph.nodes():\n    if graph.nodes()[node]["GEOID"] == \'37183017-02\':\n        print(node)\n\n#graph.nodes()[1799]\n'

In [86]:
"""
print([len(x) for x in S5])
print([len(x) for x in S28])

print(S5[0])

graph.nodes()[2475]
"""

'\nprint([len(x) for x in S5])\nprint([len(x) for x in S28])\n\nprint(S5[0])\n\ngraph.nodes()[2475]\n'

In [87]:
"""
for i in range(14):
    print(i,nx.is_connected(graph.subgraph(CONPart.parts[i])))

# 3

subgraph3 = graph.subgraph(CONPart.parts[3])


S3 = [c for c in nx.connected_components(subgraph3)]

print([len(x) for x in S3])

print(graph.nodes()[1608])

graph.nodes()[1608]["CON"] = 12

print(graph.nodes()[2433])


graph.nodes()[284]["SLDU"] = '27'
graph.nodes()[2475]["SLDU"] = '4'


#for node in graph.nodes():
#    if graph.nodes()[node]["GEOID"] == '37183017-02':
#        print(node)

#graph.nodes()[1799]
"""

'\nfor i in range(14):\n    print(i,nx.is_connected(graph.subgraph(CONPart.parts[i])))\n\n# 3\n\nsubgraph3 = graph.subgraph(CONPart.parts[3])\n\n\nS3 = [c for c in nx.connected_components(subgraph3)]\n\nprint([len(x) for x in S3])\n\nprint(graph.nodes()[1608])\n\ngraph.nodes()[1608]["CON"] = 12\n\nprint(graph.nodes()[2433])\n\n\ngraph.nodes()[284]["SLDU"] = \'27\'\ngraph.nodes()[2475]["SLDU"] = \'4\'\n\n\n#for node in graph.nodes():\n#    if graph.nodes()[node]["GEOID"] == \'37183017-02\':\n#        print(node)\n\n#graph.nodes()[1799]\n'

In [88]:
print(contiguous(CONPart))
print(contiguous(SLDUPart))
print(contiguous(SLDLPart))

False
False
True


In [89]:
ideal_con_population = sum(CONPart["population"].values()) / len(CONPart)
ideal_sldu_population = sum(SLDUPart["population"].values()) / len(SLDUPart)
ideal_sldl_population = sum(SLDLPart["population"].values()) / len(SLDLPart)

proposed_plans = [CONPart, SLDUPart, SLDLPart]
plan_names = ["CON","SLDU","SLDL"]

clist = ['green','hotpink','orange']

In [91]:
summary = [[] for plan in proposed_plans]


for i in range(len(proposed_plans)):
    #print("Dem Seats:", plan_names[i],proposed_plans[i]['PRS'].wins("Democratic"))
    print(plan_names[i])
    
    print("Cut Edges:", plan_names[i],len(proposed_plans[i]['cut_edges']))

    summary[i].append(len(proposed_plans[i]['cut_edges']))
    
    temp = 0
    for dist in proposed_plans[i].parts:
        tgraph = proposed_plans[i].subgraphs[dist]
        temp += count_spanning(tgraph)


    print("Spanning trees:",plan_names[i],temp)
    summary[i].append(temp)
    print("Mean Polsby_Popper:", plan_names[i],np.mean(list(polsby_popper(proposed_plans[i]).values())))
    summary[i].append(np.mean(list(polsby_popper(proposed_plans[i]).values())))
    print("County Splits:", plan_names[i],county_splits(proposed_plans[i]))
    summary[i].append(county_splits(proposed_plans[i]))

    
    print("Mean Population Deviation",plan_names[i],np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))
    summary[i].append(np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))

CON
Cut Edges: CON 739
Spanning trees: CON -inf
Mean Polsby_Popper: CON 0.2867798199303579
County Splits: CON 11
Mean Population Deviation CON 0.003155904760481561
SLDU
Cut Edges: SLDU 1418
Spanning trees: SLDU -inf
Mean Polsby_Popper: SLDU 0.3416265511697549
County Splits: SLDU 15
Mean Population Deviation SLDU 0.72
SLDL
Cut Edges: SLDL 2074
Spanning trees: SLDL 2657.1330931708094
Mean Polsby_Popper: SLDL 0.3765927226539025
County Splits: SLDL 36
Mean Population Deviation SLDL 0.8833333333333333


In [92]:
summary_df = pd.DataFrame(summary, columns=['Cut Edges', 'Spanning Trees', 'Mean PP', 'County Splits', 'Mean Pop Deviation'], index=plan_names)
summary_df

,Cut Edges,Spanning Trees,Mean PP,County Splits,Mean Pop Deviation
CON,739,-inf,0.286780,11,0.003156
SLDU,1418,-inf,0.341627,15,0.720000
SLDL,2074,2657.133093,0.376593,36,0.883333


In [93]:
summary_df.to_csv("./NC_Stats/NC_Compactness_Summary_ALARM.csv")

# Partisan Symmtery
Utilizes 2020 data

In [94]:
#Helper functions
def plot_symmetry_with_mean_overhaul_rb(pvec,mean,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    l=len(pvec)
    pvec = np.array(sorted(pvec))
    seats = []
    votes = []
    small = pvec[0]
    large = pvec[-1]
    
    gap = large - small
    
    
    
    #mean = np.mean(pvec)
    
    lvec = pvec - mean*np.ones([1,l])
    
    
    
    for t in range(n):
        tvec = lvec + (t/n)*np.ones([1,l])
        votes.append(t/n)#votes.append(np.mean(tvec))
        seats.append(sum(sum(tvec>=.5))/l)
        
    
 
    #print(lvec,tvec,seats[400])
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 3,color='r')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    plt.plot([mean],[sum(pvec>=.5)/l],'r*', markersize=20)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes")
    
    

    plt.show()
    
    fvotes = [1-x for x in votes]
    fseats = [1-x for x in seats]
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 1,color='blue')
    plt.plot(fvotes,fseats,linewidth = 1,color='red')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.fill_between(votes,seats,list(reversed(fseats)),color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    #plt.plot([mean],[sum(pvec>=.5)/l],'g*', markersize=10)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes: Symmetry Gaps")    
  

def plot_lots_symmetry_notmean(pvecs,means,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    
    ind = 0 
    
    clist = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown',
            'tab:pink','tab:gray','tab:olive','tab:cyan','black','lime','navy','burlywood',
            'salmon','blueviolet','chocolate','yellow','fuchsia','silver']
    plt.figure(figsize=(8,8) )  
    for pvec in pvecs: 
        l=len(pvec)
        pvec = np.array(sorted(pvec))
        seats = []
        votes = []
        small = pvec[0]
        large = pvec[-1]

        gap = large - small



        mean = means[ind] #np.mean(pvec)

        lvec = pvec - mean*np.ones([1,l])



        for t in range(n):
            tvec = lvec + (t/n)*np.ones([1,l])
            votes.append(np.mean(tvec))
            seats.append(sum(sum(tvec>=.5))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])


        cn=[str(round(float(bn), 3)) for bn in bn]
        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])

        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)

        dn=list(bn[:])
        for x in bn:
            dn.append(1-x)

        rseats=list(reversed(seats))


        en=[str(round(float(dn), 3)) for dn in dn]
        area=0
        for t in range(n):
            area += (1/n)*abs(seats[t]-(1-rseats[t])) 



         
        plt.plot(bvotes,bseats,linewidth = 2,color=clist[ind],label=enames[ind],alpha=.5)
        

        #plt.plot([.5],[.5],'ro', markersize=10)
        plt.plot([mean],[sum(pvec>=.5)/l],'o',color=clist[ind], markersize=8,zorder=100)
        
        ind +=1
    
    
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.xlabel("Democratic Vote Share")
    plt.ylabel("Democratic Seat Share")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)

    plt.xlim([xl,xu])
    plt.ylim([yl,yu])
    plt.legend()

    #plt.title("Seats -- Votes")
    
    

    plt.show()

 
def declination(vals):
    """ Compute the declination of an election.
    """
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    # Undefined if each party does not win at least one seat
    if len(Rwin) < 1 or len(Dwin) < 1:
        return False
    theta = np.arctan((1-2*np.mean(Rwin))*len(vals)/len(Rwin))
    gamma = np.arctan((2*np.mean(Dwin)-1)*len(vals)/len(Dwin))
    # Convert to range [-1,1]
    # A little extra precision just in case.
    return 2.0*(gamma-theta)/3.1415926535 


def lopsided_wins(vals):
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    
    Rmargin = abs(np.mean(Rwin)-.5)*2
    Dmargin = abs(np.mean(Dwin)-.5)*2
    
    return Rmargin - Dmargin
    

In [95]:
plan_partitions = proposed_plans
plan_part_labels = plan_names
enames = election_names

In [96]:
n_base_plans = 3

wins = [[] for i in range(n_base_plans)]
votes = [[] for i in range(n_base_plans)]
majs = [[] for i in range(n_base_plans)]
mms = [[] for i in range(n_base_plans)]
egs = [[] for i in range(n_base_plans)]
pbs =[[] for i in range(n_base_plans)]
decs = [[] for i in range(n_base_plans)]
lws = [[] for i in range(n_base_plans)]
comps = [[] for i in range(n_base_plans)]


for election in range(num_elections):
    summary = [[] for election in range(n_base_plans)]

    print(enames[election])
    #print('Votes: ', plan_partitions[0][enames[election]].percent("Democratic"))
    
    for i in range(n_base_plans):
        votes[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        summary[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        print('Votes: ', plan_partitions[i][enames[election]].percent("Democratic"))
    print('Seats')

    for i in range(n_base_plans):
        wins[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
        print(plan_part_labels[i],wins[i][-1])
        summary[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
    
    print("Majority?")
    for i in range(n_base_plans):
    
        if plan_partitions[i][enames[election]].percent("Democratic") > .5:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(1)
            else:
                majs[i].append(0)
        else:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(0)
            else:
                majs[i].append(1)
        print(plan_part_labels[i],majs[i][-1])
        summary[i].append(majs[i][-1])
    
    print("Mean-Median")
    for i in range(n_base_plans):
        mms[i].append(np.median(plan_partitions[i][enames[election]].percents("Democratic"))-plan_partitions[i][enames[election]].percent("Democratic"))
        print(plan_part_labels[i],mms[i][-1])
        summary[i].append(mms[i][-1])
        
    print("Partisan Bias")
    for i in range(n_base_plans):
        pbs[i].append(partisan_bias(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],pbs[i][-1])
        summary[i].append(pbs[i][-1])
    
    print("Efficiency Gap")
    for i in range(n_base_plans):
        egs[i].append(efficiency_gap(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],egs[i][-1])
        summary[i].append(egs[i][-1])
    
    
    print("Declination")
    for i in range(n_base_plans):
        decs[i].append(declination(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],decs[i][-1])
        summary[i].append(decs[i][-1])
        
    print("Lopsided Wins")
    for i in range(n_base_plans):
        lws[i].append(lopsided_wins(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],lws[i][-1])
        summary[i].append(lws[i][-1])
        
        
    print("Competitive 47-53 Districts")
    for i in range(n_base_plans):
        comps[i].append(np.sum([.47 < x <.53 for x in plan_partitions[i][enames[election]].percents("Democratic") ]))
        print(plan_part_labels[i],comps[i][-1])
        summary[i].append(comps[i][-1])
    
    df_summary = pd.DataFrame(summary, columns=['Votes', 'Seats', 'Majority', 'Mean-Median', 'Partisan Bias', 'Efficiency Gap', 'Declination', 'Lopsided Wins', 'Competitive Districts'], index=plan_part_labels)
    display(df_summary)
    df_summary.to_csv("./NC_Stats/NC_Partisan_Summary_"+enames[election]+"_ALARM.csv")
        

PRE
Votes:  0.49316021908904606
Votes:  0.4931602190890462
Votes:  0.4931602190890459
Seats
CON 3
SLDU 19
SLDL 50
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.05653549263009927
SLDU -0.041555580558562955
SLDL -0.042809967561940776
Partisan Bias
CON -0.2857142857142857
SLDU -0.09999999999999998
SLDL -0.08333333333333331
Efficiency Gap
CON -0.27135084028680745
SLDU -0.109172945575377
SLDL -0.07789944410543678
Declination
CON 0.6029324134800469
SLDU 0.2530927038861126
SLDL 0.17535508178247397
Lopsided Wins
CON -0.3059313668258915
SLDU -0.1260820953746138
SLDL -0.08721741741593403
Competitive 47-53 Districts
CON 0
SLDU 4
SLDL 8


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.49316,3,1,-0.056535,-0.285714,-0.271351,0.602932,-0.305931,0
SLDU,0.49316,19,1,-0.041556,-0.100000,-0.109173,0.253093,-0.126082,4
SLDL,0.49316,50,1,-0.042810,-0.083333,-0.077899,0.175355,-0.087217,8


USS
Votes:  0.49086935907488793
Votes:  0.4908693590748881
Votes:  0.4908693590748878
Seats
CON 3
SLDU 18
SLDL 48
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.050964936875558975
SLDU -0.03753370282883689
SLDL -0.038747227749330315
Partisan Bias
CON -0.2857142857142857
SLDU -0.14
SLDL -0.09166666666666667
Efficiency Gap
CON -0.26523901106305103
SLDU -0.127629288439536
SLDL -0.08842173254339156
Declination
CON 0.5814369274075682
SLDU 0.2832564338747797
SLDL 0.20013210930138162
Lopsided Wins
CON -0.2691321603269672
SLDU -0.1333159375864541
SLDL -0.09330774512369677
Competitive 47-53 Districts
CON 1
SLDU 6
SLDL 5


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.490869,3,1,-0.050965,-0.285714,-0.265239,0.581437,-0.269132,1
SLDU,0.490869,18,1,-0.037534,-0.140000,-0.127629,0.283256,-0.133316,6
SLDL,0.490869,48,1,-0.038747,-0.091667,-0.088422,0.200132,-0.093308,5


GOV
Votes:  0.5228917661889538
Votes:  0.5228917661889537
Votes:  0.5228917661889538
Seats
CON 3
SLDU 23
SLDL 55
Majority?
CON 0
SLDU 0
SLDL 0
Mean-Median
CON -0.0514588139310162
SLDU -0.04808659276558325
SLDL -0.03739276518258583
Partisan Bias
CON -0.2857142857142857
SLDU -0.09999999999999998
SLDL -0.09999999999999998
Efficiency Gap
CON -0.3304406875340655
SLDU -0.09095661605882274
SLDL -0.09433072645636087
Declination
CON 0.6743042265242205
SLDU 0.18462312454478874
SLDL 0.1909798806026596
Lopsided Wins
CON -0.4084420352192273
SLDU -0.1452702867806248
SLDL -0.15209833269871986
Competitive 47-53 Districts
CON 4
SLDU 6
SLDL 19


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.522892,3,0,-0.051459,-0.285714,-0.330441,0.674304,-0.408442,4
SLDU,0.522892,23,0,-0.048087,-0.100000,-0.090957,0.184623,-0.145270,6
SLDL,0.522892,55,0,-0.037393,-0.100000,-0.094331,0.190980,-0.152098,19
